In [1]:
# ==========================================
# 1. Environment Setup & CLIP Initialization
# ==========================================
import os
import torch
import json
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm

# Setup device (Apple MPS with fallback)
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
    
print(f"Using device: {device}")

# Initialize CLIP Model & Processor
model_name = "openai/clip-vit-base-patch32"
print(f"Loading {model_name}...")
processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name).to(device)

# Set model to evaluation mode (we are only extracting, not training)
model.eval()

# Define Dataset Directories
RAW_IMG_DIR = "../data/raw/Outfit4You/image/"
PROCESSED_DIR = "../data/processed/outfits/"

# Ensure the output directory exists
os.makedirs(PROCESSED_DIR, exist_ok=True)

Using device: mps
Loading openai/clip-vit-base-patch32...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [2]:
# ==========================================
# 2. Batch Processing Output Features (Using JSON Labels)
# ==========================================

# 1. Load Metadata
metadata_dir = '../data/raw/Outfit4You/label/'
outfits = []
for f_name in ['train.json', 'val.json', 'test.json']:
    p = os.path.join(metadata_dir, f_name)
    if os.path.exists(p):
        with open(p, 'r') as f:
            outfits.extend(json.load(f))

print(f"Found {len(outfits)} outfits in metadata.")

# Iterate over outfits with a progress bar
for outfit in tqdm(outfits, desc="Extracting CLIP Features"):
    outfit_id = outfit['id']
    
    # Collect image filenames from item_1 to item_9
    image_files = []
    for i in range(1, 10):
        item_key = f'item_{i}'
        if item_key in outfit and outfit[item_key]:
            image_files.append(outfit[item_key])
            
    if not image_files:
        print(f"Warning: No images found for outfit {outfit_id}, skipping.")
        continue
        
    item_features = []
    with torch.no_grad():
        for img_name in image_files:
            img_path = os.path.join(RAW_IMG_DIR, img_name)
            if not os.path.exists(img_path):
                # print(f"Warning: Image missing {img_path}")
                continue
            try:
                # Load image
                image = Image.open(img_path).convert("RGB")
                
                # Preprocess image
                inputs = processor(images=image, return_tensors="pt").to(device)
                
                # Extract 512-dim visual features and L2-normalize
                features = model.get_image_features(**inputs)
                if not isinstance(features, torch.Tensor):
                    features = features.pooler_output if hasattr(features, "pooler_output") else features[0]
                features = features / features.norm(p=2, dim=-1, keepdim=True)
                
                # Move to CPU, remove batch dimension (1, 512) -> (512,)
                item_features.append(features.cpu().squeeze(0))
            except Exception as e:
                print(f"Error processing {img_path}: {e}")
                
    # If we successfully extracted features from items
    if len(item_features) > 0:
        # Stack into shape: (num_items, 512)
        outfit_tensor = torch.stack(item_features)
        
        # Save to .pt file
        save_path = os.path.join(PROCESSED_DIR, f"{outfit_id}.pt")
        torch.save(outfit_tensor, save_path)
    else:
        # print(f"Warning: Failed to extract any features for {outfit_id}.")
        pass

print("CLIP Feature Extraction Complete! Tensors saved to data/processed/outfits/")


Found 15748 outfits in metadata.


Extracting CLIP Features: 100%|██████████| 15748/15748 [33:06<00:00,  7.93it/s] 

CLIP Feature Extraction Complete! Tensors saved to data/processed/outfits/
